In [ ]:
# ============================================================
# CELL 0: CHỌN CHẾ ĐỘ ABLATION VÀ SEED  (CHỈ SỬA 2 DÒNG NÀY)
# ============================================================
MODE = "shape_only"   # "vmamba_only" | "shape_only" | "shape_vmamba_no_guidance"
SEED = 4               # dùng seed 0..4 (trùng seed đã chạy với Baseline và TSVM)

ABL_MODES = ("vmamba_only", "shape_only", "shape_vmamba_no_guidance")
assert MODE in ABL_MODES, f"MODE phải là một trong {ABL_MODES}"
print(f"Ablation: MODE={MODE}, SEED={SEED}")

In [ ]:
# ============================================================
# CELL 1: SETUP REPO CUSTOM + YAML SCALE S (ĐỔI MODE TẦNG 10) + DATA_BG20.YAML
# ============================================================
import os, sys, shutil, yaml

# 1. Khai báo repo custom (bản FP32, giống 20 seed TSVM) và tập dữ liệu BG20
#    Nếu dùng tài khoản Kaggle khác: sửa KAGGLE_USER thành tên tài khoản đang giữ 2 dataset này
#    (huyphngtun: 2 dataset tải lên từ 02_ma_nguon/ultralytics_TSVM_FP32 và 01_du_lieu/Kvasir_YOLO_SEG_BG20)
KAGGLE_USER = "huyphngtun"

def _input_path(slug, sub, marker):
    # Kaggle gắn dataset ở /kaggle/input/datasets/<user>/<slug> hoặc /kaggle/input/<slug> tùy cách chạy,
    # và có thể giải nén thẳng vào gốc dataset (không còn thư mục con <sub>)
    cands = [f"{base}{tail}"
             for base in (f"/kaggle/input/datasets/{KAGGLE_USER}/{slug}", f"/kaggle/input/{slug}")
             for tail in (f"/{sub}", "")]
    return next((p for p in cands if os.path.exists(f"{p}/{marker}")), cands[0])

CUSTOM_REPO_PATH = _input_path("ultralytics-tsvm-fp32", "ultralytics_TSVM_FP32", "PATCH_FP32.md")
DATASET_ROOT = _input_path("kvasir-yolo-seg-bg20", "Kvasir_YOLO_SEG_BG20", "images/train")

assert os.path.exists(CUSTOM_REPO_PATH), f"❌ Không tìm thấy repo: {CUSTOM_REPO_PATH}"
assert os.path.exists(DATASET_ROOT), f"❌ Không tìm thấy dataset: {DATASET_ROOT}"
print("📁 Repo:", CUSTOM_REPO_PATH)
print("📁 Dữ liệu:", DATASET_ROOT)

# 2. Tạo symlink và nạp sys.path
if not os.path.exists("/kaggle/working/ultralytics"):
    os.symlink(CUSTOM_REPO_PATH, "/kaggle/working/ultralytics")
    print("🔗 Đã tạo symlink 'ultralytics'")

for p in ["/kaggle/working", CUSTOM_REPO_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

import ultralytics
from ultralytics.nn.modules import C2TSVMamba
print(f"✅ Nạp Ultralytics Custom ({ultralytics.__version__}) từ: {ultralytics.__file__}")

# 3. Tạo YAML scale 's' và đổi chế độ của tầng 10
#    Gốc: [-1, 2, C2TSVMamba, [1024]]  ->  [-1, 2, C2TSVMamba, [1024, 0.5, 16, 3, MODE]]
#    (0.5, 16, 3 là giá trị mặc định e, d_state, d_conv: chỉ MODE thay đổi)
SRC_YAML = f"{CUSTOM_REPO_PATH}/cfg/models/26/yolo26-seg-TopologyShapeVMamba.yaml"
# LƯU Ý: tên tệp PHẢI bắt đầu bằng "yolo26s" - Ultralytics đoán scale từ tên tệp, bỏ đi sẽ thành scale n
YAML_ABL = f"/kaggle/working/yolo26s-seg-TSVM-ABL-{MODE}.yaml"

with open(SRC_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg["scale"] = "s"
layer10 = cfg["backbone"][10]
assert layer10[2] == "C2TSVMamba", f"Tầng 10 không phải C2TSVMamba: {layer10}"
layer10[3] = [1024, 0.5, 16, 3, MODE]
with open(YAML_ABL, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("✅ Tầng 10:", cfg["backbone"][10])
print("✅ YAML ablation:", YAML_ABL)

# 4. Tạo file data_bg20.yaml
DATA_YAML = "/kaggle/working/data_bg20.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({
        "path": DATASET_ROOT,
        "train": "images/train",
        "val": "images/val",
        "nc": 1,
        "names": {0: "polyp"}
    }, f, sort_keys=False, allow_unicode=True)
print("✅ Đã tạo data_bg20.yaml thành công!")

In [ ]:
# ============================================================
# CELL 2: TẢI SẴN FILE TRỌNG SỐ (CÓ TỰ ĐỘNG RETRY & KIỂM TRA DUNG LƯỢNG)
# ============================================================
import os

WEIGHT_PATH = "/kaggle/working/yolo26s-seg.pt"
URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt"

if os.path.exists(WEIGHT_PATH) and os.path.getsize(WEIGHT_PATH) < 20 * 1024 * 1024:
    os.remove(WEIGHT_PATH)

if not os.path.exists(WEIGHT_PATH):
    print("⏳ Đang tải trọng số yolo26s-seg.pt...")
    !curl -L --retry 5 --retry-delay 2 -C - "{URL}" -o "{WEIGHT_PATH}"

size_mb = os.path.getsize(WEIGHT_PATH) / (1024 * 1024)
print(f" Dung lượng file: {size_mb:.2f} MB")
assert size_mb > 20.0, "❌ File tải về chưa đủ dung lượng, hãy chạy lại cell này! (Settings > Internet: ON)"

In [ ]:
# ============================================================
# CELL 3: KIỂM TRA MÔ HÌNH TRƯỚC KHI TRAIN (CHẠY NHANH, ~10 GIÂY)
# ============================================================
from ultralytics import YOLO

# Số tham số tầng 10 mong đợi (đếm trên bản FP32, scale s)
EXPECTED_L10 = {
    "vmamba_only": 1_071_873,
    "shape_only": 1_331_969,
    "shape_vmamba_no_guidance": 1_745_409,
}

_m = YOLO(YAML_ABL)
l10 = _m.model.model[10]
blk = l10.m[0]
n_l10 = sum(p.numel() for p in l10.parameters())
n_all = sum(p.numel() for p in _m.model.parameters())
print(f"Tầng 10: {type(l10).__name__}, mode khối = {blk.mode}, số khối = {len(l10.m)}")
print(f"Tham số tầng 10 = {n_l10:,} (mong đợi {EXPECTED_L10[MODE]:,}); toàn mô hình = {n_all:,}")
assert blk.mode == MODE, "❌ MODE chưa được áp vào tầng 10"
assert n_l10 == EXPECTED_L10[MODE], "❌ Số tham số tầng 10 khác mong đợi, kiểm tra lại repo"
print("✅ Mô hình ablation đúng cấu trúc")
del _m

In [ ]:
# ============================================================
# CELL 4: KHÓA TẤT ĐỊNH & TRAIN MODEL (BG20 - 100 EPOCHS) - GIỐNG HỆT 20 SEED TSVM FP32
# ============================================================
import os, random, numpy as np, torch
from ultralytics import YOLO

WORKERS = 2
BATCH = 8
EPOCHS = 100

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=False)

model = YOLO(YAML_ABL).load(WEIGHT_PATH)

RUN_NAME = f"Kvasir_BG20_YOLO26s_seg_ABL_{MODE}_s{SEED}_w{WORKERS}"
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    cache=False,
    amp=False,
    optimizer="AdamW",
    seed=SEED,
    lr0=0.001,
    warmup_epochs=5.0,
    deterministic=True,
    patience=100,
    close_mosaic=10,
    project="/kaggle/working/runs",
    name=RUN_NAME,
    exist_ok=True,
    plots=True
)

In [ ]:
# ============================================================
# CELL 5: NÉN KẾT QUẢ ĐỂ TẢI VỀ
# ============================================================
!cd /kaggle/working && zip -r "runs_{RUN_NAME}.zip" runs